## TP Séquence 4 - Consommation des ménages de Lomé ##

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path


## PARTIE 1 - Charger et contrôler

In [12]:
DOSSIER = r"D:\cours Licence 3\Traitement de donnees\data"
CSV = os.path.join(DOSSIER, "consommation_menages_lome.csv")

df = pd.read_csv(CSV)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   quartier          200 non-null    object 
 1   service           200 non-null    object 
 2   periode_couverte  200 non-null    object 
 3   ancien_index      200 non-null    float64
 4   nouvel_index      197 non-null    float64
 5   menage            200 non-null    object 
dtypes: float64(2), object(4)
memory usage: 9.5+ KB


In [18]:
#Q1 : 
print("\nQ1 - Lignes, colonnes :", df.shape)
#Unité d'observation : UN RELEVÉ de compteur (un ménage x un service x une période,avec plusieurs relevés possibles par mois). 


Q1 - Lignes, colonnes : (200, 6)


In [19]:
# Q2 : nunique
print("\nQ2 - Valeurs distinctes :")
print(df[["quartier", "menage", "service", "periode_couverte"]].nunique())
print("\nRelevés par service :")
print(df["service"].value_counts())


Q2 - Valeurs distinctes :
quartier             3
menage              11
service              2
periode_couverte     3
dtype: int64

Relevés par service :
service
eau            111
electricite     89
Name: count, dtype: int64


In [20]:
# Q3 : manquants et doublons
print("\nQ3 - Valeurs manquantes par colonne :")
print(df.isna().sum())
print("Lignes en double :", df.duplicated().sum())


Q3 - Valeurs manquantes par colonne :
quartier            0
service             0
periode_couverte    0
ancien_index        0
nouvel_index        3
menage              0
dtype: int64
Lignes en double : 1


In [21]:
# Q4 : conso et négatifs
df = df.drop_duplicates().copy()
df["conso"] = df["nouvel_index"] - df["ancien_index"]
negatifs = df[df["conso"] < 0]
print("\nQ4 - Relevés à conso négative (compteur remplacé) :", len(negatifs))
print(negatifs)


Q4 - Relevés à conso négative (compteur remplacé) : 2
   quartier      service periode_couverte  ancien_index  nouvel_index  menage  \
38       Bè  electricite        Mars 2026        7549.0          38.0  MEN-A4   
90     Agoè  electricite     Janvier 2026        4934.0           5.0  MEN-C2   

     conso  
38 -7511.0  
90 -4929.0  


In [25]:
# Q5 : df_ok / df_exclus
masque_exclu = df["nouvel_index"].isna() | (df["conso"] < 0)
df_exclus = df[masque_exclu]
df_ok = df[~masque_exclu].copy()
print("\nQ5 - Lignes gardées :", len(df_ok), "| exclues :", len(df_exclus))
print(df_exclus)
# CONTRÔLE 1 : aucune ligne ne disparaît sans explication
assert len(df_ok) + len(df_exclus) == len(df)


Q5 - Lignes gardées : 194 | exclues : 5
    quartier      service periode_couverte  ancien_index  nouvel_index  \
38        Bè  electricite        Mars 2026        7549.0          38.0   
90      Agoè  electricite     Janvier 2026        4934.0           5.0   
125       Bè  electricite     Février 2026        7301.0           NaN   
167     Agoè          eau     Janvier 2026         789.1           NaN   
191   Tokoin          eau     Février 2026         549.7           NaN   

     menage   conso  
38   MEN-A4 -7511.0  
90   MEN-C2 -4929.0  
125  MEN-A4     NaN  
167  MEN-C2     NaN  
191  MEN-B4     NaN  


## PARTIE 2 - Consommation et groupby

In [26]:
# Q1 : synthese
synthese = (
    df_ok
    .groupby(["quartier", "service"], as_index=False)
    .agg(
        conso_totale=("conso", "sum"),
        nb_releves=("conso", "size"),
        nb_menages=("menage", "nunique"),
    )
)
print(synthese)
 
# CONTRÔLE 2 : la somme après groupby = la somme avant
assert np.isclose(df_ok["conso"].sum(), synthese["conso_totale"].sum())

  quartier      service  conso_totale  nb_releves  nb_menages
0     Agoè          eau          89.4          30           3
1     Agoè  electricite         620.0          18           2
2       Bè          eau         185.1          42           4
3       Bè  electricite        1446.0          37           4
4   Tokoin          eau         167.4          37           4
5   Tokoin  electricite        1222.0          30           3


In [27]:
# Q2 : unité d'observation
print("\nQ2 - Lignes de synthese :", len(synthese))
# Nouvelle unité : un couple (quartier, service). 3 quartiers x 2 services = 6 lignes.


Q2 - Lignes de synthese : 6


In [28]:
# Q3 : quartier le plus consommateur par service
print("\nQ3 - Quartier le plus consommateur :")
for s in ["eau", "electricite"]:
    sub = synthese[synthese["service"] == s].sort_values("conso_totale", ascending=False)
    print(f"  {s} : {sub.iloc[0]['quartier']} ({sub.iloc[0]['conso_totale']:.1f})")


Q3 - Quartier le plus consommateur :
  eau : Bè (185.1)
  electricite : Bè (1446.0)


In [29]:
# Q4 : par service et période
par_periode = (
    df_ok.groupby(["service", "periode_couverte"])["conso"].sum()
    .unstack("periode_couverte")[ORDRE_MOIS]
)
print("\nQ4 - Conso totale par service et période :")
print(par_periode.round(1))


Q4 - Conso totale par service et période :
periode_couverte  Janvier 2026  Février 2026  Mars 2026
service                                                
eau                      144.5         144.7      152.7
electricite             1085.0        1083.0     1120.0


## "PARTIE 3 - Choisir le dénominateur"

In [30]:
# Q1 : colonnes de ratio pour l'eau
eau_syn = synthese[synthese["service"] == "eau"].copy()
eau_syn["conso_par_releve"] = eau_syn["conso_totale"] / eau_syn["nb_releves"]
eau_syn["conso_par_menage"] = eau_syn["conso_totale"] / eau_syn["nb_menages"]
eau_syn["conso_par_menage_mois"] = eau_syn["conso_par_menage"] / 3
print(eau_syn.round(2))

  quartier service  conso_totale  nb_releves  nb_menages  conso_par_releve  \
0     Agoè     eau          89.4          30           3              2.98   
2       Bè     eau         185.1          42           4              4.41   
4   Tokoin     eau         167.4          37           4              4.52   

   conso_par_menage  conso_par_menage_mois  
0             29.80                   9.93  
2             46.28                  15.43  
4             41.85                  13.95  


In [31]:
# Q2 : classements
print("\nQ2 - Classement par conso_par_releve :")
print(eau_syn.sort_values("conso_par_releve", ascending=False)[["quartier", "conso_par_releve", "nb_releves"]].round(2))
print("Classement par conso_par_menage :")
print(eau_syn.sort_values("conso_par_menage", ascending=False)[["quartier", "conso_par_menage", "nb_menages"]].round(2))


Q2 - Classement par conso_par_releve :
  quartier  conso_par_releve  nb_releves
4   Tokoin              4.52          37
2       Bè              4.41          42
0     Agoè              2.98          30
Classement par conso_par_menage :
  quartier  conso_par_menage  nb_menages
2       Bè             46.28           4
4   Tokoin             41.85           4
0     Agoè             29.80           3


In [32]:
# Q3 : électricité, ménages abonnés vs effectif réel
elec_syn = synthese[synthese["service"] == "electricite"].copy()
effectif_reel = {"Tokoin": 4, "Bè": 4, "Agoè": 3}
elec_syn["effectif_reel"] = elec_syn["quartier"].map(effectif_reel)
elec_syn["par_abonne"] = elec_syn["conso_totale"] / elec_syn["nb_menages"]
elec_syn["par_menage_quartier"] = elec_syn["conso_totale"] / elec_syn["effectif_reel"]
print("\nQ3 - Électricité : abonnés (nunique) vs effectif réel")
print(elec_syn[["quartier", "conso_totale", "nb_menages", "effectif_reel", "par_abonne", "par_menage_quartier"]].round(1))


Q3 - Électricité : abonnés (nunique) vs effectif réel
  quartier  conso_totale  nb_menages  effectif_reel  par_abonne  \
1     Agoè         620.0           2              3       310.0   
3       Bè        1446.0           4              4       361.5   
5   Tokoin        1222.0           3              4       407.3   

   par_menage_quartier  
1                206.7  
3                361.5  
5                305.5  


## Phrase à compléter : « Diviser par le nombre de lignes mesure la consommation par relevé ; diviser par nunique(menage) mesure la consommation par ménage (sur la période). »

# PARTIE 4 - Moyenne pondérée (eau)

In [34]:
# Q1
eau = df_ok[df_ok["service"] == "eau"]
menage_stats = (
    eau.groupby(["quartier", "menage"], as_index=False)
    .agg(somme=("conso", "sum"), effectif=("conso", "size"), moyenne=("conso", "mean"))
)
print(menage_stats.round(3))

   quartier  menage  somme  effectif  moyenne
0      Agoè  MEN-C1   28.0        11    2.545
1      Agoè  MEN-C2   26.6         9    2.956
2      Agoè  MEN-C3   34.8        10    3.480
3        Bè  MEN-A1   34.5        11    3.136
4        Bè  MEN-A2   50.2        12    4.183
5        Bè  MEN-A3   45.5        10    4.550
6        Bè  MEN-A4   54.9         9    6.100
7    Tokoin  MEN-B1   42.7         9    4.744
8    Tokoin  MEN-B2   37.7         9    4.189
9    Tokoin  MEN-B3   57.3        10    5.730
10   Tokoin  MEN-B4   29.7         9    3.300


In [35]:
 #Q2 : méthode fausse vs exacte, par quartier
comp = menage_stats.groupby("quartier").apply(
    lambda g: pd.Series({
        "moyenne_simple (fausse)": g["moyenne"].mean(),
        "moyenne_ponderee (exacte)": g["somme"].sum() / g["effectif"].sum(),
    }),
    include_groups=False,
)
comp["ecart"] = comp["moyenne_simple (fausse)"] - comp["moyenne_ponderee (exacte)"]
print("\nQ2 :")
print(comp.round(4))


Q2 :
          moyenne_simple (fausse)  moyenne_ponderee (exacte)   ecart
quartier                                                            
Agoè                       2.9937                     2.9800  0.0137
Bè                         4.4924                     4.4071  0.0853
Tokoin                     4.4908                     4.5243 -0.0335


In [36]:
# Q3 : vérification avec np.average
q = comp.index[0]
g = menage_stats[menage_stats["quartier"] == q]
print(f"\nQ3 - {q} : np.average = {np.average(g['moyenne'], weights=g['effectif']):.4f}"
      f" | somme/effectif = {g['somme'].sum() / g['effectif'].sum():.4f}")


Q3 - Agoè : np.average = 2.9800 | somme/effectif = 2.9800


In [37]:
# Q4 : niveau des 3 quartiers
quartier_stats = eau.groupby("quartier").agg(somme=("conso", "sum"), effectif=("conso", "size"))
quartier_stats["moyenne"] = quartier_stats["somme"] / quartier_stats["effectif"]
moy_des_moy = quartier_stats["moyenne"].mean()
moy_globale = eau["conso"].mean()
print(f"\nQ4 - Moyenne des 3 moyennes locales : {moy_des_moy:.4f}")
print(f"     Moyenne par relevé sur tout le fichier : {moy_globale:.4f}")
print(f"     Écart : {moy_des_moy - moy_globale:.4f}")


Q4 - Moyenne des 3 moyennes locales : 3.9705
     Moyenne par relevé sur tout le fichier : 4.0541
     Écart : -0.0836


## Q5 : la moyenne n'est pas additive. On ne peut pas recombiner des moyennes de blocs sans leurs poids. Le couple (somme, effectif) est additif, donc on peut le cumuler d'un bloc à l'autre et diviser une seule fois à la fin.

# PARTIE 5 - pivot_table et melt

In [38]:
# Q1
tab = pd.pivot_table(
    eau, values="conso", index="quartier", columns="periode_couverte",
    aggfunc="sum", fill_value=0, margins=True, margins_name="Total",
)
# CONTRÔLE 3 : total du pivot = somme brute
assert np.isclose(tab.loc["Total", "Total"], eau["conso"].sum())

In [39]:
# Q2 : ordre chronologique des colonnes
tab = tab[ORDRE_MOIS + ["Total"]]
print("Q1-Q2 - Eau :")
print(tab.round(1))

Q1-Q2 - Eau :
periode_couverte  Janvier 2026  Février 2026  Mars 2026  Total
quartier                                                      
Agoè                      28.6          28.8       32.0   89.4
Bè                        59.9          62.3       62.9  185.1
Tokoin                    56.0          53.6       57.8  167.4
Total                    144.5         144.7      152.7  441.9


In [40]:
# Q3 : df.pivot lèverait une ValueError
try:
    eau.pivot(index="quartier", columns="periode_couverte", values="conso")
except ValueError as e:
    print("\nQ3 - df.pivot -> ValueError :", e)
# pivot exige un couple (index, colonne) unique ; ici plusieurs lignes par quartier x mois.
# pivot_table règle le problème avec aggfunc="sum".


Q3 - df.pivot -> ValueError : Index contains duplicate entries, cannot reshape


In [41]:
# Q4 : électricité
elec = df_ok[df_ok["service"] == "electricite"]
tab_elec = pd.pivot_table(
    elec, values="conso", index="quartier", columns="periode_couverte",
    aggfunc="sum", fill_value=0, margins=True, margins_name="Total",
)[ORDRE_MOIS + ["Total"]]
assert np.isclose(tab_elec.loc["Total", "Total"], elec["conso"].sum())
print("\nQ4 - Électricité :")
print(tab_elec.round(1))


Q4 - Électricité :
periode_couverte  Janvier 2026  Février 2026  Mars 2026   Total
quartier                                                       
Agoè                     181.0         211.0      228.0   620.0
Bè                       501.0         448.0      497.0  1446.0
Tokoin                   403.0         424.0      395.0  1222.0
Total                   1085.0        1083.0     1120.0  3288.0


In [42]:
# Q5 : large puis long
eau_large = pd.pivot_table(
    eau, values="conso", index="menage", columns="periode_couverte",
    aggfunc="sum", fill_value=0,
)[ORDRE_MOIS]
print("\nQ5 - eau_large :")
print(eau_large.round(1))
 
eau_long = (
    eau_large.reset_index()
    .melt(id_vars="menage", var_name="periode_couverte", value_name="conso")
)


Q5 - eau_large :
periode_couverte  Janvier 2026  Février 2026  Mars 2026
menage                                                 
MEN-A1                    11.5          11.2       11.8
MEN-A2                    17.0          15.3       17.9
MEN-A3                    15.9          14.5       15.1
MEN-A4                    15.5          21.3       18.1
MEN-B1                    13.5          13.3       15.9
MEN-B2                    13.4          12.1       12.2
MEN-B3                    17.7          19.4       20.2
MEN-B4                    11.4           8.8        9.5
MEN-C1                    10.2           8.3        9.5
MEN-C2                     6.8           9.7       10.1
MEN-C3                    11.6          10.8       12.4


In [43]:
# Q6 : contrôles
print("\nQ6 - len(eau_long) =", len(eau_long), "(attendu", 11 * 3, ")")
assert len(eau_long) == 11 * 3
# CONTRÔLE 4 : la somme est conservée après melt
assert np.isclose(eau_long["conso"].sum(), eau["conso"].sum())
print("Somme avant pivot :", round(eau["conso"].sum(), 2), "| après melt :", round(eau_long["conso"].sum(), 2))
 
# Cellules à 0 dans eau_large : vraie conso nulle ou donnée absente ?
combos = eau.groupby(["menage", "periode_couverte"]).size().unstack(fill_value=0)
print("\nCellules (ménage, mois) sans relevé valide :")
print((combos == 0).stack()[lambda s: s])


Q6 - len(eau_long) = 33 (attendu 33 )
Somme avant pivot : 441.9 | après melt : 441.9

Cellules (ménage, mois) sans relevé valide :
Series([], dtype: bool)


# PARTIE 6 - Synthèse chiffrée

In [44]:
tot_eau = eau_syn.sort_values("conso_totale", ascending=False).iloc[0]
par_men = eau_syn.sort_values("conso_par_menage", ascending=False).iloc[0]
print(f"Constat 1 : total eau max = {tot_eau['quartier']} ({tot_eau['conso_totale']:.1f} m3, {int(tot_eau['nb_releves'])} relevés)")
print(f"            par ménage max = {par_men['quartier']} ({par_men['conso_par_menage']:.1f} m3/ménage, {int(par_men['nb_menages'])} ménages)")
print("Constat 2 :"); print(comp.round(4))
print("Constat 3 :"); print(par_periode.round(1))
print("\nLes 4 assert de conservation sont passés.")

Constat 1 : total eau max = Bè (185.1 m3, 42 relevés)
            par ménage max = Bè (46.3 m3/ménage, 4 ménages)
Constat 2 :
          moyenne_simple (fausse)  moyenne_ponderee (exacte)   ecart
quartier                                                            
Agoè                       2.9937                     2.9800  0.0137
Bè                         4.4924                     4.4071  0.0853
Tokoin                     4.4908                     4.5243 -0.0335
Constat 3 :
periode_couverte  Janvier 2026  Février 2026  Mars 2026
service                                                
eau                      144.5         144.7      152.7
electricite             1085.0        1083.0     1120.0

Les 4 assert de conservation sont passés.


## Quartier le plus consommateur d'eau : Bè, avec 185,1 m³ au total (42 relevés) et 46,3 m³ par ménage (4 ménages), soit 15,4 m³ par ménage et par mois. Contrôle : la somme des 3 quartiers fait 441,9 m³.
## Moyenne simple contre moyenne pondérée à Bè : 4,492 contre 4,407 m³ par relevé, soit un écart de +0,085 (environ +1,9 %).
## Évolution de janvier à mars : l'eau augmente de 5,7 % (144,5 → 152,7 m³) et l'électricité de 3,2 % (1085 → 1120 kWh). Contrôle : les totaux des tableaux croisés sont 441,9 m³ et 3288 kWh.